# RAG quick start

Search this project's local index from a notebook.

**No `ipywidgets` anywhere in this notebook, on purpose.** VS Code caches widget state per
notebook file path, and stale registrations make plots and tables render several times.
Plain output and `Markdown` avoid that entirely.


## 1. Open the index

`Index.find()` walks up from the working directory to the nearest `.rag/`.
Models load lazily, so this cell is instant.


In [1]:
import sys
from pathlib import Path

# --- kernel guard -----------------------------------------------------------
# This notebook MUST run on the vault's own venv (<vault>/.venv). Selecting any
# other kernel imports rag_toolkit fine (it is pure-python and on sys.path below)
# and then dies much later on `import sentence_transformers`, which is confusing.
# Fail here instead, with the fix.
try:
    import sentence_transformers  # noqa: F401
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "Wrong kernel.\n"
        f"  running on : {sys.executable}\n"
        "  expected   : <vault>/.venv/bin/python\n"
        "\n"
        "Pick the kernel named 'my-wiki RAG (.venv)':\n"
        "  VS Code  -> 'Select Kernel' (top right) -> Jupyter Kernel... -> my-wiki RAG (.venv)\n"
        "  Jupyter  -> Kernel -> Change Kernel -> my-wiki RAG (.venv)\n"
        "\n"
        "If it is not listed, register it once:\n"
        "  <vault>/.venv/bin/python -m ipykernel install --user \\\n"
        "      --name my-wiki-rag --display-name 'my-wiki RAG (.venv)'"
    ) from exc

# Put the vendored toolkit on the path. The installer writes a rag_toolkit.pth,
# but macOS flags files under a dot-directory in an iCloud container UF_HIDDEN,
# and CPython 3.13+ skips hidden .pth files — so do not rely on it.
_toolkit = Path.cwd()
while _toolkit != _toolkit.parent and not (_toolkit / ".rag" / "toolkit").is_dir():
    _toolkit = _toolkit.parent
_toolkit = str(_toolkit / ".rag" / "toolkit")
if _toolkit not in sys.path:
    sys.path.insert(0, _toolkit)

from rag_toolkit import Index, to_markdown
from IPython.display import Markdown

index = Index.find()          # or Index.at('/path/to/project')
status = index.status()

print(f"kernel    : {sys.executable}")
print(f"workspace : {status['rag_dir']}")
print(f"indexed   : {status.get('files', 0)} files, {status.get('chunks', 0)} chunks")
print(f"model     : {status['embedding'].get('model')}")
if status.get('warning'):
    print(f"\nWARNING: {status['warning']}")

/Users/Khaled.Alabsi/.local/share/rag/my-wiki/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


kernel    : /Users/Khaled.Alabsi/.local/share/rag/my-wiki/venv/bin/python
workspace : /Users/Khaled.Alabsi/Library/Mobile Documents/iCloud~md~obsidian/Documents/my-wiki/.rag
indexed   : 144 files, 3055 chunks
model     : BAAI/bge-m3


## 2. Search

The first search loads the embedding model, so it is slower than the ones after it.


In [2]:
hits = index.search('worked example of MYT decomposition on Tennessee Eastman data', k=5)

for hit in hits:
    print(f"{hit['score']:>8.4f}  {hit['citation']}")

The Transformer `cache_dir` argument is deprecated. Please pass `cache_dir` via `model_kwargs`, `processor_kwargs`, and/or `config_kwargs` instead.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 50541.57it/s]
/Users/Khaled.Alabsi/Library/Mobile Documents/iCloud~md~obsidian/Documents/my-wiki/.rag/toolkit/rag_toolkit/embed.py:108: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._dimension = int(self._model.get_sentence_embedding_dimension() or 0)


  0.6664  PhD/myt-decomposition.md:65-86 — MYT Decomposition (Mason, Young & Tracy, 1995) > Worked TEP Example
  0.1985  PhD/Noise Handling in Statistical and Multivariate Process Monitoring_ A Literature Review.md:3-8 — Noise Handling in Statistical Process Monitoring (SPM) and Multivariate Statistical Process Control (MSPC): A Literature Review > TL;DR
  0.1890  PhD/myt-decomposition.md:87-96 — MYT Decomposition (Mason, Young & Tracy, 1995) > The Core Limitation
  0.1575  PhD/myt-decomposition.md:3-11 — MYT Decomposition (Mason, Young & Tracy, 1995) > Table of Contents
  0.1460  PhD/myt-decomposition.md:16-26 — MYT Decomposition (Mason, Young & Tracy, 1995) > The Formula


## 3. Read the passages

`to_markdown` renders hits with their citations. Nothing is truncated silently —
`max_chars` controls it explicitly.


In [3]:
Markdown(to_markdown(hits, max_chars=600))

**1. PhD/myt-decomposition.md:65-86 — MYT Decomposition (Mason, Young & Tracy, 1995) > Worked TEP Example** — score `0.666405` (rerank)

> ## Worked TEP Example
> 
> Baseline statistics from NOC training data, and a faulty sample (e.g. a cooling-water-related fault):
> 
> | Variable | Normal mean $\mu_j$ | Normal std dev $\sigma_j$ | Faulty value $x_j$ | Deviation $d_j = x_j-\mu_j$ | Contribution $d_j^2/\sigma_j^2$ |
> |---|---|---|---|---|---|
> | Reactor Temp | 120.4 | 0.3 | 121.7 | 1.3 | 18.8 |
> | Reactor Pressure | 2705 | 13 | 2718 | 13 | 1.0 |
> | CW Outlet Temp | 94.6 | 1.5 | 98.9 | 4.3 | 8.2 |
> | CW Flow | 41.1 | 2.0 | 46.0 | 4.9 | 6.0 |
> | Reactor Level | 75.0 | 1.8 | 75.3 | 0.3 | 0.03 |
> 
> **Ranking by contribution:** Reactor Temp (18.8) >…

**2. PhD/Noise Handling in Statistical and Multivariate Process Monitoring_ A Literature Review.md:3-8 — Noise Handling in Statistical Process Monitoring (SPM) and Multivariate Statistical Process Control (MSPC): A Literature Review > TL;DR** — score `0.198506` (rerank)

> ## TL;DR
> 
> - **The state of the art splits into two paradigms.** *Noise isolation/characterization* uses latent-variable projection (PCA, PLS, ICA, factor analysis, probabilistic PCA), multiscale wavelet decomposition, and robust statistics to separate a “noise subspace” from a “signal subspace”; *noise reduction/filtering* uses classical smoothing (moving average, median, Savitzky–Golay), model-based filters (Kalman, particle), transform-domain denoising (wavelet, Fourier, EMD/SSA), and modern deep denoising (denoising/variational autoencoders, diffusion models). Most methods still implicitly…

**3. PhD/myt-decomposition.md:87-96 — MYT Decomposition (Mason, Young & Tracy, 1995) > The Core Limitation** — score `0.188965` (rerank)

> ## The Core Limitation
> 
> Sum the contributions above: $18.8 + 1.0 + 8.2 + 6.0 + 0.03 \approx 34$. This does **not** equal the true $T^2$ for the sample, because MYT throws away every cross-covariance term in $\Sigma^{-1}$.
> 
> Concrete illustration with two correlated variables (correlation strength 0.8, meaning they normally move together): if both deviate by the same amount, $d = (2, 2)$:
> - True $T^2 \approx 4.4$ — small, because moving together is *normal* for these two variables.
> - MYT contributions: $4$ and $4$, summing to $8$ — nearly double the true value.
> 
> **Why:** MYT has no concept of "t…

**4. PhD/myt-decomposition.md:3-11 — MYT Decomposition (Mason, Young & Tracy, 1995) > Table of Contents** — score `0.157548` (rerank)

> ## Table of Contents
> - [[#What Problem MYT Solves|What Problem MYT Solves]]
> - [[#The Formula|The Formula]]
> - [[#Why This Formula, Specifically|Why This Formula, Specifically]]
> - [[#Why Divide Instead of Subtract|Why Divide Instead of Subtract]]
> - [[#Worked TEP Example|Worked TEP Example]]
> - [[#The Core Limitation|The Core Limitation]]
> - [[#Summary Table|Summary Table]]

**5. PhD/myt-decomposition.md:16-26 — MYT Decomposition (Mason, Young & Tracy, 1995) > The Formula** — score `0.145963` (rerank)

> ## The Formula
> 
> $$\text{contrib}_j = \frac{d_j^2}{\sigma_j^2}$$
> 
> Where:
> - $j$ = index of the variable (1 through 52 for TEP)
> - $d_j$ = deviation of variable $j$, defined as $d_j = x_j - \mu_j$, where $x_j$ is the variable's current (possibly faulty) value and $\mu_j$ is its normal average value (learned from NOC — normal operating condition — training data)
> - $\sigma_j$ = the normal standard deviation of variable $j$ (how much it naturally fluctuates under normal operation), so $\sigma_j^2$ is its variance
> 
> In plain terms: square the deviation, divide by the variable's own normal variance. Thi…


## 4. Filter

Filters narrow *before* ranking, so weaker matches get a chance to surface.


In [4]:
hits = index.search(
    'T2 fault attribution',
    k=10,
    ext='.md',             # this vault indexes .md and .json only
    path='PhD/**',         # glob, relative to the vault root
    # source='my-wiki',    # the one configured source
    # since='2026-07-01',  # only notes modified since
)

for hit in hits:
    print(f"{hit['matched_by']:<8} {hit['score']:>8.4f}  {hit['citation']}")

rerank     0.8806  PhD/pca-t2-spe-attribution-methods.md:37-57 — PCA T² / SPE Attribution — Two Base Methods > Method 1 — PCA Loading Attribution (T²)
rerank     0.8738  PhD/hawkins-decomposition-t2-fault-diagnosis.md:88-102 — Hawkins' Whitened T² Decomposition for Fault Diagnosis > From Raw Measurements to Contributions — Step by Step
rerank     0.8349  PhD/pca-t2-spe-attribution-methods.md:81-134 — PCA T² / SPE Attribution — Two Base Methods > Worked Numerical Example (4 variables, 2 components)
rerank     0.8204  PhD/pca-t2-spe-attribution-methods.md:135-142 — PCA T² / SPE Attribution — Two Base Methods > Why the Two Methods Give Different Answers
rerank     0.7985  PhD/pca-t2-spe-attribution-methods.md:11-36 — PCA T² / SPE Attribution — Two Base Methods > Setup and Symbols
rerank     0.7400  PhD/pca-t2-spe-attribution-methods.md:58-80 — PCA T² / SPE Attribution — Two Base Methods > Method 2 — PCA Residual Attribution (SPE)
rerank     0.7255  PhD/pca-t2-spe-attribution-methods.md:3-

## 5. As a DataFrame

Useful for scanning many results at once. Needs pandas; skip this cell if it is absent.


In [5]:
# pandas is not part of the .rag environment; fall back to plain output if absent.
rows = [
    {'score': h['score'], 'matched_by': h['matched_by'],
     'citation': h['citation'], 'chars': len(h['text'])}
    for h in hits
]
try:
    import pandas as pd
    frame = pd.DataFrame(rows)
    display(frame)
except ModuleNotFoundError:
    for r in rows:
        print(f"{r['score']:>8.4f}  {r['matched_by']:<8} {r['chars']:>5}  {r['citation']}")

  0.8806  rerank     838  PhD/pca-t2-spe-attribution-methods.md:37-57 — PCA T² / SPE Attribution — Two Base Methods > Method 1 — PCA Loading Attribution (T²)
  0.8738  rerank     671  PhD/hawkins-decomposition-t2-fault-diagnosis.md:88-102 — Hawkins' Whitened T² Decomposition for Fault Diagnosis > From Raw Measurements to Contributions — Step by Step
  0.8349  rerank    1354  PhD/pca-t2-spe-attribution-methods.md:81-134 — PCA T² / SPE Attribution — Two Base Methods > Worked Numerical Example (4 variables, 2 components)
  0.8204  rerank     712  PhD/pca-t2-spe-attribution-methods.md:135-142 — PCA T² / SPE Attribution — Two Base Methods > Why the Two Methods Give Different Answers
  0.7985  rerank    1735  PhD/pca-t2-spe-attribution-methods.md:11-36 — PCA T² / SPE Attribution — Two Base Methods > Setup and Symbols
  0.7400  rerank     946  PhD/pca-t2-spe-attribution-methods.md:58-80 — PCA T² / SPE Attribution — Two Base Methods > Method 2 — PCA Residual Attribution (SPE)
  0.7255  rerank 

## 6. Build a prompt context block

`context_block` returns the passages pre-formatted with citations attached — ready to
paste into a prompt. The index retrieves; whatever you hand this to writes the answer.


In [6]:
print(index.context_block('Geeignetheitserklärung', k=6, max_chars=1200))

# Retrieved for: Geeignetheitserklärung

## [1] Banking/mifid-wphg-banking-notes.md:107-136 — Geeignetheitserklärung (GEE)  (score 0.952476)
# Geeignetheitserklärung (GEE)

**The GEE explains why the bank's recommendation is suitable for the
customer.**

Created after the suitability assessment.

Contains: - Recommended product(s) - Investment objectives - Risk
profile - Financial situation - Knowledge & experience - Investment
horizon - Explanation of suitability - Warnings - Advisor information

Typical architecture:

``` text
Customer Profile
      │
Risk Profile
      │
Knowledge & Experience
      │
Target Market
      │
Recommendation Engine
      │
      ▼
GEE Generator
      ▼
PDF
```

## [2] Banking/COBA/AVD/knowledge.md:15-28 — Business Knowledge Extraction Report > 3. Key Business Concepts & Glossary  (score 0.905963)
## 3. Key Business Concepts & Glossary
| Term | Business Meaning |
|------|------------------|
| **AVD / FRÜHSTART** | Early Retirement Savings Account (minors

## 7. Inspect one chunk in full

`anchor` is the raw citation data: line range, page, sheet and rows, or cell.


In [7]:
if hits:
    top = hits[0]
    print('path        :', top['path'])
    print('heading path:', top['heading_path'])
    print('anchor      :', top['anchor'])
    print('-' * 70)
    print(top['text'])

path        : /Users/Khaled.Alabsi/Library/Mobile Documents/iCloud~md~obsidian/Documents/my-wiki/PhD/pca-t2-spe-attribution-methods.md
heading path: PCA T² / SPE Attribution — Two Base Methods > Method 1 — PCA Loading Attribution (T²)
anchor      : {'line_start': 37, 'line_end': 57}
----------------------------------------------------------------------
## Method 1 — PCA Loading Attribution (T²)

**What it measures:** how much each variable drove the sample too far along directions the model already knows about.

**Step 1 — scores.** Project $x$ into the $c$-dim space:

$$t_k = \sum_{j=1}^{p} P_{j,k} \, x_j, \qquad k = 1,...,c$$

**Step 2 — standardize.**

$$z_k = \frac{t_k}{\sigma_k}$$

**Step 3 — attribute back to variables.**

$$\text{contrib}_j = \sum_{k=1}^{c} |z_k| \cdot |P_{j,k}|$$

Each term multiplies "how abnormal is component $k$" ($|z_k|$) by "how much does variable $j$ feed component $k$" ($|P_{j,k}|$), summed over all $c$ components. Variables with large loadings on the mo

## 8. What did the pipeline actually do?

`search_report` exposes the counts behind a search: how many came from dense retrieval,
how many from keyword search, whether reranking ran. This is the first thing to look at
when results are worse than expected.


In [8]:
report = index.search_report('how do I set a boundary when scope creeps mid-task', k=5)

print(f"dense hits : {report.dense_count}")
print(f"text hits  : {report.text_count}")
print(f"after fuse : {report.fused_count}")
print(f"reranked   : {report.reranked}")
for note in report.notes:
    print(f"note       : {note}")

dense hits : 60
text hits  : 60
after fuse : 107
reranked   : True


## 9. Close

Releases the store. `Index` is also a context manager if you prefer `with Index.find() as index:`.


In [9]:
index.close()

---

**Updating the index is a shell command, not a notebook cell.** Indexing is heavy and
long-running, and a notebook kernel is the wrong place for it:

```bash
rag update          # after adding or editing files
rag status          # confirm it moved
rag doctor          # when something looks wrong
```
